# MathEBT — DINO training on Colab

Phase 1: DINO-only training on the certified-view corpus (see `docs/DESIGN.md`). Code is pulled directly from GitHub (`Baptistecaille/lean-dino`).

Lean extraction itself has no Colab equivalent (no `lake`/`lean` toolchain here) and was done locally — but everything downstream of it is built **in this notebook**: the raw corpus (`data/raw_full`, one JSONL per Mathlib module) is hosted as a private dataset on the Hugging Face Hub, and the tokenizer + train/valid/test splits are regenerated from it by `scripts/train_tokenizer.py` and `scripts/build_splits.py`. Nothing pre-built is uploaded by hand.

**Before running anything**: Runtime → Change runtime type → GPU (T4 is fine to start).

**Why Drive matters here**: Colab's local VM disk is wiped on disconnect. Checkpoints are written to Drive so that if the runtime disconnects mid-run, re-running the training cell resumes from `last.pt` instead of losing everything (see `DinoTrainer.load` — auto-resume was added specifically for this). The built tokenizer/splits are also cached to Drive so a reconnect doesn't require re-downloading the 2GB+ raw corpus and rebuilding them from scratch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/lean-dino'  # where checkpoints/outputs/data cache persist
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive project dir:', DRIVE_DIR)

## Get the code — clone from GitHub

Public repo, no auth needed. Re-running this cell later (e.g. after a disconnect) pulls the latest commit.

In [ ]:
import os

PROJECT_DIR = '/content/lean-dino'
REPO_URL = 'https://github.com/Baptistecaille/lean-dino.git'

if os.path.exists(PROJECT_DIR):
    print('repo already present, pulling latest')
    !git -C "$PROJECT_DIR" pull
else:
    !git clone "$REPO_URL" "$PROJECT_DIR"

%cd /content/lean-dino

In [ ]:
%cd /content/lean-dino
!pip install -q -e . 2>&1 | tail -20

## Get the data — build tokenizer + splits from the raw corpus

`data/raw_full` (the Lean extraction output, ~167k declarations across ~8300 Mathlib modules) lives in the private dataset **`albertreporter1/math-ebt-mathlib-raw`** on the Hugging Face Hub, packed as a single `raw_full.tar.gz` — it's the one artifact with no Colab equivalent, so it's hosted rather than regenerated here. (Downloading it as ~8300 loose files instead of one archive hits HF's per-file rate limit hard — that's why it's one tarball, not a directory of JSONL files.)

Everything after that step is reproducible in-notebook: `scripts/train_tokenizer.py` and `scripts/build_splits.py` both read `data/raw_full` directly (see `configs/dino_v0.yaml`), so running them here regenerates `data/tokenizer/bpe.json` and `data/splits/{train,valid,test}.jsonl` exactly as they'd be built locally.

You'll need a Hugging Face token with at least read access to the dataset (create one at https://huggingface.co/settings/tokens, or reuse one you already have) — paste it into the widget the next cell pops up. It is **not** printed or logged.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import os
import shutil
import subprocess
import tarfile

from huggingface_hub import hf_hub_download

RAW_DATASET_REPO = 'albertreporter1/math-ebt-mathlib-raw'
DATA_CACHE = f'{DRIVE_DIR}/data_cache'  # cached tokenizer + splits, so reconnects skip the rebuild
os.makedirs(DATA_CACHE, exist_ok=True)

tokenizer_cached = os.path.exists(f'{DATA_CACHE}/tokenizer/bpe.json')
splits_cached = os.path.exists(f'{DATA_CACHE}/splits/train.jsonl')


def run_or_show(cmd):
    """subprocess.run(check=True) swallows the child's traceback in Colab's cell
    output ordering -- capture explicitly and print it before raising."""
    result = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        result.check_returncode()


if tokenizer_cached and splits_cached:
    print('found tokenizer + splits cached on Drive -- reusing, skipping raw download and rebuild')
    for name in ['tokenizer', 'splits']:
        dst = f'{PROJECT_DIR}/data/{name}'
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(f'{DATA_CACHE}/{name}', dst)
else:
    print('no cache on Drive -- downloading raw corpus archive from HF and building tokenizer + splits')
    archive_path = hf_hub_download(
        repo_id=RAW_DATASET_REPO, repo_type='dataset', filename='raw_full.tar.gz',
    )
    os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
    raw_target = f'{PROJECT_DIR}/data/raw_full'
    if os.path.exists(raw_target):
        shutil.rmtree(raw_target)
    with tarfile.open(archive_path, 'r:gz') as tf:
        tf.extractall(f'{PROJECT_DIR}/data')
    print(len(os.listdir(raw_target)), 'module files at', raw_target)

    run_or_show(['python', 'scripts/train_tokenizer.py', '--config', 'configs/dino_v0.yaml'])
    run_or_show(['python', 'scripts/build_splits.py', '--config', 'configs/dino_v0.yaml'])

    for name in ['tokenizer', 'splits']:
        shutil.copytree(f'{PROJECT_DIR}/data/{name}', f'{DATA_CACHE}/{name}')
    print('cached tokenizer + splits to Drive for future sessions')

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Runtime > Change runtime type')

In [ ]:
# sanity check before committing to a long run -- same tests as run locally
!python -m pytest tests/ -q

## Train

`--output-dir` points at Drive, so checkpoints (and `metrics.jsonl`, which `eval_all.py`'s stability gate reads) persist. If the runtime disconnects, just re-run this cell — `train_dino.py` auto-resumes from `last.pt` if it exists in `output_dir`.

This is the `certified` arm (the config default). `total_steps=100000` at the config's effective batch (256) is a genuinely long run — expect this to take hours even on GPU. Watch the logged `teacher_entropy_mean` / `teacher_max_prob` early on (collapse shows up in the first few thousand steps if it's going to happen — see docs/DESIGN.md §11).

In [ ]:
OUTPUT_DIR = f'{DRIVE_DIR}/outputs/dino_v0_certified'
!python scripts/train_dino.py --config configs/dino_v0.yaml --output-dir "$OUTPUT_DIR"

## Evaluate

Run after training reaches `gates.min_stable_steps` (50000 by default) — earlier than that the stability gate will just fail for lack of data, which is expected, not a bug.

In [ ]:
!python scripts/eval_all.py --config configs/dino_v0.yaml --ckpt "$OUTPUT_DIR/last.pt"

## Optional: the three ablation arms

The actual result the project exists to produce (`certified − dropout_only`, docs/DESIGN.md §6). Same compute budget for all three. This will take roughly 3x as long as the single run above — only run this once the single `certified` run above looks healthy, so you're not burning GPU time three times over on a broken config.

In [ ]:
for mode in ['dropout_only', 'naive', 'certified']:
    out_dir = f'{DRIVE_DIR}/outputs/dino_v0_certified_{mode}'
    print(f'=== {mode} -> {out_dir} ===')
    !python scripts/train_dino.py --config configs/dino_v0.yaml --view-mode {mode} --output-dir "$out_dir"
    !python scripts/eval_all.py --config configs/dino_v0.yaml --ckpt "$out_dir/last.pt" || true